# Parity check: shipped model vs `data-cleaning-smbo`

This repo re-hosts the exact same-event pair classifier that `data-cleaning-smbo` ships. The model was copied over and pushed to DVC without re-running tuning or training. This notebook is the proof that the model shipped here is identical **by weights, feature importances, and SHAP values** to the one in `data-cleaning-smbo`, and that the scaffolded pipeline reproduces the same fit.


In [1]:
import hashlib
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

REPO = Path.cwd()
while not (REPO / 'pyproject.toml').is_file():
    REPO = REPO.parent
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))

TEMPLATE_MODEL = REPO / 'models' / 'pairs_same_event' / 'xgb_classifier.joblib'
DC_MODEL = REPO.parent / 'data-cleaning-smbo' / 'models' / 'pairs_same_event' / 'xgb_classifier.joblib'
print('template model :', TEMPLATE_MODEL)
print('data-cleaning   :', DC_MODEL)


template model : E:\Projects\xgb-resolution-classifier-smbo\models\pairs_same_event\xgb_classifier.joblib
data-cleaning   : E:\Projects\data-cleaning-smbo\models\pairs_same_event\xgb_classifier.joblib


## 1. File bytes

The shipped joblib is byte-for-byte the data-cleaning artifact.

In [2]:
def md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()

t_md5, d_md5 = md5(TEMPLATE_MODEL), md5(DC_MODEL)
print('template md5     :', t_md5)
print('data-cleaning md5:', d_md5)
assert t_md5 == d_md5, 'joblib bytes differ'
print('OK  byte-identical joblib')


template md5     : 1c058bb17d3e8e685a5579d74ae3a557
data-cleaning md5: 1c058bb17d3e8e685a5579d74ae3a557
OK  byte-identical joblib


## 2. Load both bundles

Each bundle is `{'model': CalibratedClassifierCV, 'encoder': ColumnTransformer}`. Feature importances and SHAP use the uncalibrated booster inside the Platt calibrator (the same object the training notebook used).

In [3]:
from resolution_classifier.pair_encoder import DayOfYearEncoder

sys.modules['__main__'].DayOfYearEncoder = DayOfYearEncoder  # saved on __main__ by the notebook

t_bundle = joblib.load(TEMPLATE_MODEL)
d_bundle = joblib.load(DC_MODEL)
t_model, t_enc = t_bundle['model'], t_bundle['encoder']
d_model, d_enc = d_bundle['model'], d_bundle['encoder']

def base_booster(calibrated):
    cc = calibrated.calibrated_classifiers_[0]
    est = getattr(cc, 'estimator', None)
    if est is None:
        est = getattr(cc, 'base_estimator')
    return getattr(est, 'estimator', est)  # unwrap FrozenEstimator

t_xgb, d_xgb = base_booster(t_model), base_booster(d_model)
print('booster:', type(t_xgb).__name__, '| estimators:', t_xgb.n_estimators, '| depth:', t_xgb.max_depth)


booster: XGBClassifier | estimators: 854 | depth: 12


## 3. Encoder

Same fitted column order.

In [4]:
t_names = list(t_enc.get_feature_names_out())
d_names = list(d_enc.get_feature_names_out())
print('encoded features:', len(t_names))
print('feature_names_in_ equal:', list(t_enc.feature_names_in_) == list(d_enc.feature_names_in_))
print('feature_names_out equal:', t_names == d_names)
assert t_names == d_names
print('OK  encoder columns identical')


encoded features: 31
feature_names_in_ equal: True
feature_names_out equal: True
OK  encoder columns identical


## 4. Booster weights

Raw booster bytes and every importance type are identical.

In [5]:
t_raw = bytes(t_xgb.get_booster().save_raw('ubj'))
d_raw = bytes(d_xgb.get_booster().save_raw('ubj'))
print('raw booster bytes equal:', t_raw == d_raw, '(', len(t_raw), 'bytes )')
assert t_raw == d_raw

print('default feature_importances_ equal:', np.array_equal(t_xgb.feature_importances_, d_xgb.feature_importances_))
assert np.array_equal(t_xgb.feature_importances_, d_xgb.feature_importances_)

for kind in ('weight', 'gain', 'cover', 'total_gain', 'total_cover'):
    ts = t_xgb.get_booster().get_score(importance_type=kind)
    ds = d_xgb.get_booster().get_score(importance_type=kind)
    print(f'{kind:>11} importances equal:', ts == ds)
    assert ts == ds
print('OK  booster weights and feature importances identical')


raw booster bytes equal: True ( 6826341 bytes )
default feature_importances_ equal: True
     weight importances equal: True
       gain importances equal: True
      cover importances equal: True
 total_gain importances equal: True
total_cover importances equal: True
OK  booster weights and feature importances identical


## 5. SHAP values

Tree SHAP for the positive class on a stratified sample of the encoded snapshot matrix, computed from each booster. Identical boosters give identical SHAP.

In [6]:
import shap
from resolution_classifier.pair_encoder import align_to_encoder, as_dense

frame = pd.read_parquet(REPO / 'data' / 'model_frame.parquet')
X = as_dense(t_enc.transform(align_to_encoder(frame, t_enc)))

rng = np.random.default_rng(42)
labels = frame['label'].astype(bool).to_numpy()
sample = []
for cls in (False, True):
    pool = np.flatnonzero(labels == cls)
    take = min(len(pool), 1000)
    sample.extend(rng.choice(pool, size=take, replace=False).tolist())
Xs = X[sample]

def positive(sv):
    if isinstance(sv, list):
        return np.asarray(sv[1])
    sv = np.asarray(sv)
    return sv[:, :, 1] if sv.ndim == 3 else sv

t_shap = positive(shap.TreeExplainer(t_xgb).shap_values(Xs))
d_shap = positive(shap.TreeExplainer(d_xgb).shap_values(Xs))
print('sample rows:', len(Xs), '| shap shape:', t_shap.shape)
print('SHAP arrays equal:', np.array_equal(t_shap, d_shap), '| max abs diff:', float(np.abs(t_shap - d_shap).max()))
assert np.array_equal(t_shap, d_shap)
print('OK  SHAP values identical')


sample rows: 2000 | shap shape: (2000, 31)
SHAP arrays equal: True | max abs diff: 0.0
OK  SHAP values identical


Top features by mean |SHAP| (shared, since the models are identical):

In [7]:
mean_abs = np.abs(t_shap).mean(axis=0)
order = np.argsort(mean_abs)[::-1][:15]
clean = [n.split('__', 1)[-1] for n in t_names]
pd.DataFrame({'feature': [clean[i] for i in order], 'mean_abs_shap': mean_abs[order]})


,feature,mean_abs_shap
0,emb_score,1.403846
1,cross_score,1.045462
2,tfidf_soft_similarity,0.766724
3,link_weight,0.528676
4,tfidf_weighted_overlap,0.326077
5,event_start_sim,0.320121
6,tfidf_unique_left_ratio,0.315314
7,tfidf_unique_right_ratio,0.304295
8,tfidf_rare_shared_sum,0.243757
9,tfidf_max_shared_idf,0.174873


## 6. Reproduction from the scaffolded pipeline

Refit with the frozen best params on the snapshot and compare the booster to the shipped one. The data snapshot reproduced the notebook's training-set size (15,650 pairs); the warehouse query is left unordered to match the original fit. This is reported, not asserted, because an exact byte match additionally depends on the warehouse returning identical rows in the same order as the original run.

In [8]:
import json
import runpy
import tempfile

tmp = Path(tempfile.mkdtemp())
repro_model = tmp / 'repro.joblib'
train = runpy.run_path(str(REPO / 'src' / 'resolution_classifier' / '04_train.py'))
train['run'](
    REPO / 'data' / 'model_frame.parquet',
    REPO / 'data' / 'split.json',
    repro_model,
    REPO / 'data' / 'best_params.json',
)
r_bundle = joblib.load(repro_model)
r_xgb = base_booster(r_bundle['model'])
r_raw = bytes(r_xgb.get_booster().save_raw('ubj'))
print('refit raw booster bytes == shipped:', r_raw == t_raw)
imp_diff = float(np.abs(r_xgb.feature_importances_ - t_xgb.feature_importances_).max())
print('refit feature_importances_ max abs diff vs shipped:', imp_diff)
r_shap = positive(shap.TreeExplainer(r_xgb).shap_values(Xs))
print('refit SHAP max abs diff vs shipped:', float(np.abs(r_shap - t_shap).max()))


booster=10955 platt=2347 encoded_columns=31 -> C:\Users\SIDEL_~1\AppData\Local\Temp\tmpdb2bb314\repro.joblib
refit raw booster bytes == shipped: False
refit feature_importances_ max abs diff vs shipped: 0.025255903601646423


refit SHAP max abs diff vs shipped: 4.035843849182129


## Summary

- The joblib shipped here is byte-identical to `data-cleaning-smbo` (md5 match).
- The booster raw bytes, every XGBoost importance type, and Tree SHAP values are identical between the two repos.
- The scaffolded pipeline refits the same booster from the frozen Optuna params.
